# Otto RecSys - MBSRec (Multi-Behavioral Sequential Recommendation)

## Metrics:
- **Recall@10, Recall@20**
- **NDCG@10, NDCG@20**
- **HR@10, HR@20** (Hit Rate)

Paper: https://arxiv.org/pdf/2312.09684v1

In [1]:
# Install PyTorch
!pip install torch polars pandas -q

In [2]:
import os
import time
import glob
import random
import math
import numpy as np
import polars as pl
import pandas as pd
from collections import defaultdict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

Device: cuda
PyTorch version: 2.10.0+cu128


In [3]:
# Configuration
SEED = 42

class Config:
    embed_dim = 64
    batch_size = 256
    lr = 0.001
    maxlen = 50
    num_epochs = 30
    num_heads = 2
    num_layers = 2
    dropout = 0.3
    warmup_steps = 500

config = Config()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

OUTPUT_DIR = "/kaggle/working/mbsrec_output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOG_FILE = os.path.join(OUTPUT_DIR, "training_log.txt")

In [4]:
# Data paths
DATA_PATH = '/kaggle/input/datasets/cdeotte/otto-validation/'

train_files = sorted(glob.glob(os.path.join(DATA_PATH, 'train_parquet/*.parquet')))
test_files = sorted(glob.glob(os.path.join(DATA_PATH, 'test_parquet/*.parquet')))

print(f"Found {len(train_files)} train parquet files")
print(f"Found {len(test_files)} test parquet files")

Found 100 train parquet files
Found 20 test parquet files


In [5]:
# Load data
print("Loading data...")

train_dfs = [pl.read_parquet(f) for f in train_files]
train_df = pl.concat(train_dfs)

test_df = pl.concat([pl.read_parquet(f) for f in test_files])

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Loading data...
Train shape: (163955180, 4)
Test shape: (7683577, 4)


In [6]:
# Partition data
type_map = {'clicks': 0, 'carts': 1, 'orders': 2}
type_weight_map = {0: 0.1, 1: 0.3, 2: 0.9}

def partition_otto_data(df):
    user_train = {}
    user_valid = {}
    user_test = {}
    Beh = {}
    Beh_w = {}
    
    df = df.sort(['session', 'ts'])
    
    usernum = 0
    itemnum = 0
    
    df_pd = df.to_pandas()
    for session, group in df_pd.groupby('session'):
        session = int(session)
        usernum = max(usernum, session)
        items = group['aid'].tolist()
        types = group['type'].tolist()
        
        for aid in items:
            itemnum = max(itemnum, aid)
        
        for aid, t in zip(items, types):
            beh_vec = [1 if t == 'clicks' else 0,
                       1 if t == 'carts' else 0,
                       1 if t == 'orders' else 0]
            Beh[(session, aid)] = beh_vec
            Beh_w[(session, aid)] = type_weight_map.get(type_map.get(t, 0), 0.1)
        
        n = len(items)
        if n >= 3:
            user_train[session] = items[:-2]
            user_valid[session] = [items[-2]]
            user_test[session] = [items[-1]]
        elif n == 2:
            user_train[session] = [items[0]]
            user_valid[session] = [items[1]]
            user_test[session] = []
        else:
            user_train[session] = items
            user_valid[session] = []
            user_test[session] = []
    
    Beh[(0, 0)] = [0, 0, 0]
    Beh_w[(0, 0)] = 0
    
    return user_train, user_valid, user_test, Beh, Beh_w, usernum, itemnum

print("Partitioning data...")
user_train, user_valid, user_test, Beh, Beh_w, usernum, itemnum = partition_otto_data(train_df)

print(f"Users: {usernum}, Items: {itemnum}")
print(f"Train sessions: {len(user_train)}, Valid sessions: {len(user_valid)}, Test sessions: {len(user_test)}")

Partitioning data...
Users: 11098527, Items: 1855602
Train sessions: 11098528, Valid sessions: 11098528, Test sessions: 11098528


In [7]:
# MBSRec Model in PyTorch

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"
        
        self.q_linear = nn.Linear(hidden_dim, hidden_dim)
        self.k_linear = nn.Linear(hidden_dim, hidden_dim)
        self.v_linear = nn.Linear(hidden_dim, hidden_dim)
        self.out_linear = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None, causal=True):
        batch_size = query.size(0)
        
        Q = self.q_linear(query).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_linear(key).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_linear(value).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        if causal:
            seq_len = query.size(1)
            causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=query.device), diagonal=1).bool()
            scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_dim)
        output = self.out_linear(context)
        
        return output


class FeedForward(nn.Module):
    def __init__(self, hidden_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(hidden_dim, ff_dim)
        self.linear2 = nn.Linear(ff_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = F.relu(self.linear1(x))
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(hidden_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.ff = FeedForward(hidden_dim, ff_dim, dropout)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        attn_out = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x


class MBSRecModel(nn.Module):
    def __init__(self, num_items, num_behaviors, embed_dim, maxlen, num_heads, num_layers, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.maxlen = maxlen
        
        self.item_embedding = nn.Embedding(num_items + 1, embed_dim, padding_idx=0)
        self.behavior_embedding = nn.Linear(num_behaviors, embed_dim, bias=False)
        self.pos_encoding = PositionalEncoding(embed_dim, maxlen, dropout)
        
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, embed_dim * 4, dropout)
            for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(embed_dim)
        
        self.init_weights()
    
    def init_weights(self):
        nn.init.xavier_uniform_(self.item_embedding.weight)
        self.item_embedding.weight.data[0] = 0
    
    def forward(self, seq, beh_context):
        seq = seq.long()
        
        item_emb = self.item_embedding(seq)
        beh_emb = self.behavior_embedding(beh_context)
        
        x = item_emb + beh_emb
        x = self.pos_encoding(x)
        
        mask = (seq != 0).unsqueeze(1).unsqueeze(2)
        
        for block in self.transformer_blocks:
            x = block(x, mask)
        
        x = self.layer_norm(x)
        
        return x


class MBSRec(nn.Module):
    def __init__(self, num_items, num_behaviors=3, embed_dim=64, maxlen=50, num_heads=2, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = MBSRecModel(num_items, num_behaviors, embed_dim, maxlen, num_heads, num_layers, dropout)
        self.item_embedding = self.encoder.item_embedding
    
    def forward(self, seq, beh_context):
        enc_out = self.encoder(seq, beh_context)
        return enc_out

print("MBSRec Model defined")

MBSRec Model defined


In [8]:
# Dataset and Sampler

def random_neq(l, r, s):
    t = np.random.randint(l, r)
    while t in s:
        t = np.random.randint(l, r)
    return t


class MBSRecDataset(Dataset):
    def __init__(self, user_train, Beh, Beh_w, usernum, itemnum, maxlen, Beh_num=3):
        self.user_train = user_train
        self.Beh = Beh
        self.Beh_w = Beh_w
        self.usernum = usernum
        self.itemnum = itemnum
        self.maxlen = maxlen
        self.Beh_num = Beh_num
        self.users = list(user_train.keys())
    
    def __len__(self):
        return len(self.users)
    
    def __getitem__(self, idx):
        user = self.users[idx]
        
        if len(self.user_train[user]) <= 1:
            return self.__getitem__((idx + 1) % len(self.users))
        
        seq = np.zeros(self.maxlen, dtype=np.int32)
        pos = np.zeros(self.maxlen, dtype=np.int32)
        neg = np.zeros(self.maxlen, dtype=np.int32)
        beh_seq = np.zeros((self.maxlen, self.Beh_num), dtype=np.float32)
        beh_pos = np.zeros((self.maxlen, self.Beh_num), dtype=np.float32)
        pos_weight = np.zeros(self.maxlen, dtype=np.float32)
        
        nxt = self.user_train[user][-1]
        idx_pos = self.maxlen - 1
        
        rated = set(self.user_train[user])
        
        for i in reversed(self.user_train[user][:-1]):
            seq[idx_pos] = i
            pos[idx_pos] = nxt
            neg[idx_pos] = random_neq(1, self.itemnum + 1, rated)
            
            beh_seq[idx_pos] = self.Beh.get((user, i), [0, 0, 0])
            beh_pos[idx_pos] = self.Beh.get((user, nxt), [0, 0, 0])
            pos_weight[idx_pos] = self.Beh_w.get((user, nxt), 0.1)
            
            nxt = i
            idx_pos -= 1
            if idx_pos == -1:
                break
        
        return (
            torch.LongTensor(seq),
            torch.LongTensor(pos),
            torch.LongTensor(neg),
            torch.FloatTensor(beh_seq),
            torch.FloatTensor(beh_pos),
            torch.FloatTensor(pos_weight)
        )

print("Dataset class defined")

Dataset class defined


In [9]:
# Loss Function

class WeightedBPRLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, pos_logits, neg_logits, pos_weight, mask):
        pos_weight = pos_weight * mask
        
        loss = -torch.log(torch.sigmoid(pos_logits - neg_logits) + 1e-24) * pos_weight
        loss = loss.sum() / (mask.sum() + 1e-24)
        
        return loss

print("Loss function defined")

Loss function defined


In [10]:
# Evaluation Metrics

def compute_dcg(scores, k):
    scores = scores[:k]
    dcg = sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(scores))
    return dcg


def compute_ndcg(rel_items, pred_items, k):
    scores = [1 if item in rel_items else 0 for item in pred_items[:k]]
    ideal_scores = [1] * min(len(rel_items), k) + [0] * max(0, k - len(rel_items))
    idcg = compute_dcg(ideal_scores, k)
    if idcg == 0:
        return 0.0
    dcg = compute_dcg(scores, k)
    return dcg / idcg


def compute_recall(rel_items, pred_items, k):
    if len(rel_items) == 0:
        return 0.0
    pred_k = set(pred_items[:k])
    rel_set = set(rel_items)
    hits = len(pred_k & rel_set)
    return hits / len(rel_set)


def compute_hit_rate(rel_items, pred_items, k):
    pred_k = set(pred_items[:k])
    rel_set = set(rel_items)
    return 1.0 if len(pred_k & rel_set) > 0 else 0.0


def evaluate_full(model, user_train, user_valid, Beh, itemnum, config, device, ks=[10, 20]):
    model.eval()
    
    metrics = {
        'Recall@10': 0.0, 'Recall@20': 0.0,
        'NDCG@10': 0.0, 'NDCG@20': 0.0,
        'HR@10': 0.0, 'HR@20': 0.0
    }
    valid_users = 0
    
    users = list(user_train.keys())
    if len(users) > 2000:
        users = random.sample(users, 2000)
    
    Beh_num = 3
    
    with torch.no_grad():
        for u in users:
            if u not in user_train or len(user_train[u]) < 1:
                continue
            if u not in user_valid or len(user_valid[u]) < 1:
                continue
            
            seq = np.zeros(config.maxlen, dtype=np.int32)
            beh_seq = np.zeros((config.maxlen, Beh_num), dtype=np.float32)
            idx_pos = config.maxlen - 1
            
            for i in reversed(user_train[u]):
                seq[idx_pos] = i
                beh_seq[idx_pos] = Beh.get((u, i), [0, 0, 0])
                idx_pos -= 1
                if idx_pos == -1:
                    break
            
            rel_items = user_valid[u]
            rated = set(user_train[u])
            
            candidates = rel_items.copy()
            test_item_cxt = [Beh.get((u, rel_items[0]), [0, 0, 0])]
            
            for _ in range(99):
                t = np.random.randint(1, itemnum + 1)
                while t in rated:
                    t = np.random.randint(1, itemnum + 1)
                candidates.append(t)
                test_item_cxt.append([0, 0, 0])
            
            seq_tensor = torch.LongTensor(seq).unsqueeze(0).to(device)
            beh_tensor = torch.FloatTensor(beh_seq).unsqueeze(0).to(device)
            
            enc_out = model(seq_tensor, beh_tensor)
            seq_emb = enc_out[:, -1, :]
            
            cand_tensor = torch.LongTensor(candidates).to(device)
            cand_emb = model.encoder.item_embedding(cand_tensor)
            
            scores = torch.matmul(seq_emb, cand_emb.T).squeeze().cpu().numpy()
            ranked_indices = scores.argsort()[::-1]
            pred_items = [candidates[i] for i in ranked_indices]
            
            for k in ks:
                metrics[f'Recall@{k}'] += compute_recall(rel_items, pred_items, k)
                metrics[f'NDCG@{k}'] += compute_ndcg(rel_items, pred_items, k)
                metrics[f'HR@{k}'] += compute_hit_rate(rel_items, pred_items, k)
            
            valid_users += 1
    
    if valid_users > 0:
        for key in metrics:
            metrics[key] /= valid_users
    
    return metrics, valid_users

print("Evaluation metrics defined")

Evaluation metrics defined


In [11]:
# Initialize Model
print("\n" + "="*70)
print("Initializing MBSRec Model")
print("="*70 + "\n")

model = MBSRec(
    num_items=itemnum,
    num_behaviors=3,
    embed_dim=config.embed_dim,
    maxlen=config.maxlen,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    dropout=config.dropout
).to(DEVICE)

dataset = MBSRecDataset(user_train, Beh, Beh_w, usernum, itemnum, config.maxlen)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=config.lr, total_steps=len(dataloader) * config.num_epochs,
    pct_start=0.1
)
criterion = WeightedBPRLoss()
scaler = GradScaler()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Users: {usernum}, Items: {itemnum}")
print(f"Embed dim: {config.embed_dim}, Heads: {config.num_heads}, Layers: {config.num_layers}")
print(f"Maxlen: {config.maxlen}, Batch size: {config.batch_size}, Epochs: {config.num_epochs}")


Initializing MBSRec Model

Model parameters: 118,858,880
Users: 11098527, Items: 1855602
Embed dim: 64, Heads: 2, Layers: 2
Maxlen: 50, Batch size: 256, Epochs: 30


/tmp/ipykernel_43/1307486565.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [12]:
# Training Loop
T = 0.0
t0 = time.time()
best_metrics = {'Recall@10': 0.0, 'Recall@20': 0.0, 'NDCG@10': 0.0, 'NDCG@20': 0.0, 'HR@10': 0.0, 'HR@20': 0.0}

log_file = open(LOG_FILE, 'w')
log_file.write("="*80 + "\n")
log_file.write("MBSRec Training Log - Otto Validation Dataset\n")
log_file.write("Metrics: Recall@10, Recall@20, NDCG@10, NDCG@20, HR@10, HR@20\n")
log_file.write("="*80 + "\n\n")
log_file.flush()

print("\n" + "="*70)
print("Starting Training")
print("="*70)
print("\nMetrics: Recall@10, Recall@20, NDCG@10, NDCG@20, HR@10, HR@20\n")

for epoch in range(1, config.num_epochs + 1):
    model.train()
    total_loss = 0
    epoch_start = time.time()
    
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{config.num_epochs}")
    for seq, pos, neg, beh_seq, beh_pos, pos_weight in pbar:
        seq = seq.to(DEVICE)
        pos = pos.to(DEVICE)
        neg = neg.to(DEVICE)
        beh_seq = beh_seq.to(DEVICE)
        beh_pos = beh_pos.to(DEVICE)
        pos_weight = pos_weight.to(DEVICE)
        
        mask = (seq != 0).float()
        
        optimizer.zero_grad()
        
        with autocast():
            enc_out = model(seq, beh_seq)
            
            seq_emb = enc_out.view(-1, config.maxlen, config.embed_dim)
            pos_emb = model.encoder.item_embedding(pos)
            neg_emb = model.encoder.item_embedding(neg)
            
            batch_size, seq_len, emb_dim = seq_emb.shape
            seq_emb_flat = seq_emb.reshape(batch_size * seq_len, emb_dim)
            pos_emb_flat = pos_emb.reshape(batch_size * seq_len, emb_dim)
            neg_emb_flat = neg_emb.reshape(batch_size * seq_len, emb_dim)
            
            pos_logits = (seq_emb_flat * pos_emb_flat).sum(dim=-1)
            neg_logits = (seq_emb_flat * neg_emb_flat).sum(dim=-1)
            
            mask_flat = mask.reshape(-1)
            pos_weight_flat = pos_weight.reshape(-1)
            
            loss = criterion(pos_logits, neg_logits, pos_weight_flat, mask_flat)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(dataloader)
    epoch_time = time.time() - epoch_start
    T += epoch_time
    
    print(f"\n--- Epoch {epoch} Evaluation ---")
    print(f"Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s")
    
    # Evaluate
    metrics, n_users = evaluate_full(model, user_train, user_valid, Beh, itemnum, config, DEVICE, ks=[10, 20])
    
    log_line = f"Epoch {epoch}/{config.num_epochs} | "
    log_line += f"Loss: {avg_loss:.4f} | "
    log_line += f"R@10: {metrics['Recall@10']:.4f} | R@20: {metrics['Recall@20']:.4f} | "
    log_line += f"NDCG@10: {metrics['NDCG@10']:.4f} | NDCG@20: {metrics['NDCG@20']:.4f} | "
    log_line += f"HR@10: {metrics['HR@10']:.4f} | HR@20: {metrics['HR@20']:.4f} | "
    log_line += f"Time: {T:.2f}s"
    
    print(log_line)
    log_file.write(log_line + "\n")
    log_file.flush()
    
    # Save best model
    if metrics['NDCG@20'] > best_metrics['NDCG@20']:
        best_metrics = metrics.copy()
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': best_metrics,
        }, os.path.join(OUTPUT_DIR, 'mbsrec_best.pth'))
        print(f"  ** Best model saved! NDCG@20: {best_metrics['NDCG@20']:.4f} **")

log_file.write("\n" + "="*80 + "\n")
log_file.write(f"Training Complete! Best Metrics:\n")
for k, v in best_metrics.items():
    log_file.write(f"  {k}: {v:.4f}\n")
log_file.write("="*80 + "\n")
log_file.close()

print("\n" + "="*70)
print("Training Complete!")
print("="*70)
print("\nBest Metrics:")
for k, v in best_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\nLog saved to: {LOG_FILE}")


Starting Training

Metrics: Recall@10, Recall@20, NDCG@10, NDCG@20, HR@10, HR@20



Epoch 1/30:   0%|          | 0/43354 [00:00<?, ?it/s]/tmp/ipykernel_43/3303758424.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_43/3303758424.py:59: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Epoch 1/30: 100%|██████████| 43354/43354 [10:58<00:00, 65.79it/s, loss=nan]



--- Epoch 1 Evaluation ---
Training Loss: nan, Time: 658.96s
Epoch 1/30 | Loss: nan | R@10: 0.0042 | R@20: 0.9536 | NDCG@10: 0.0020 | NDCG@20: 0.2584 | HR@10: 0.0042 | HR@20: 0.9536 | Time: 658.96s
  ** Best model saved! NDCG@20: 0.2584 **


Epoch 2/30: 100%|██████████| 43354/43354 [09:36<00:00, 75.25it/s, loss=nan]



--- Epoch 2 Evaluation ---
Training Loss: nan, Time: 576.12s
Epoch 2/30 | Loss: nan | R@10: 0.0090 | R@20: 0.9504 | NDCG@10: 0.0039 | NDCG@20: 0.2581 | HR@10: 0.0090 | HR@20: 0.9504 | Time: 1235.08s


...


--- Epoch 28 Evaluation ---
Training Loss: nan, Time: 577.94s
Epoch 28/30 | Loss: nan | R@10: 0.0042 | R@20: 0.9399 | NDCG@10: 0.0019 | NDCG@20: 0.2547 | HR@10: 0.0042 | HR@20: 0.9399 | Time: 17179.28s


Epoch 29/30: 100%|██████████| 43354/43354 [09:41<00:00, 74.61it/s, loss=nan]



--- Epoch 29 Evaluation ---
Training Loss: nan, Time: 581.05s
Epoch 29/30 | Loss: nan | R@10: 0.0084 | R@20: 0.9476 | NDCG@10: 0.0039 | NDCG@20: 0.2575 | HR@10: 0.0084 | HR@20: 0.9476 | Time: 17760.33s


Epoch 30/30: 100%|██████████| 43354/43354 [10:17<00:00, 70.16it/s, loss=nan]



--- Epoch 30 Evaluation ---
Training Loss: nan, Time: 617.96s
Epoch 30/30 | Loss: nan | R@10: 0.0037 | R@20: 0.9490 | NDCG@10: 0.0021 | NDCG@20: 0.2574 | HR@10: 0.0037 | HR@20: 0.9490 | Time: 18378.29s

Training Complete!

Best Metrics:
  Recall@10: 0.0063
  Recall@20: 0.9549
  NDCG@10: 0.0039
  NDCG@20: 0.2601
  HR@10: 0.0063
  HR@20: 0.9549

Log saved to: /kaggle/working/mbsrec_output/training_log.txt


In [13]:
# Load best model
checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'mbsrec_best.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Best model loaded")

Best model loaded


In [14]:
# Generate Recommendations
print("\n--- Generating Recommendations for Submission ---")

test_df = pl.concat([pl.read_parquet(f) for f in test_files])
print(f"Test data shape: {test_df.shape}")

session_histories = test_df.sort(['session', 'ts']).group_by('session').agg([
    pl.col('aid').alias('items')
])
session_dict = {row['session']: row['items'] for row in session_histories.iter_rows(named=True)}

print(f"Test sessions: {len(session_dict)}")


--- Generating Recommendations for Submission ---
Test data shape: (7683577, 4)
Test sessions: 1801251


In [15]:
# Generate predictions
recommendations = {}
Beh_num = 3

for session in tqdm(list(session_dict.keys())[:50000], desc="Generating recommendations"):
    items = session_dict[session]
    if len(items) == 0:
        recommendations[session] = []
        continue
    
    seq = np.zeros(config.maxlen, dtype=np.int32)
    beh_seq = np.zeros((config.maxlen, Beh_num), dtype=np.float32)
    idx_pos = config.maxlen - 1
    
    for i in reversed(items):
        if idx_pos < 0:
            break
        seq[idx_pos] = i
        beh_seq[idx_pos] = Beh.get((session, i), [0, 0, 0])
        idx_pos -= 1
    
    rated = set(items)
    candidates = list(items)[:20]
    test_cxt = [Beh.get((session, i), [0, 0, 0]) for i in candidates]
    
    for _ in range(80):
        t = np.random.randint(1, min(itemnum + 1, 200000))
        while t in rated:
            t = np.random.randint(1, min(itemnum + 1, 200000))
        candidates.append(t)
        test_cxt.append([0, 0, 0])
    
    with torch.no_grad():
        seq_tensor = torch.LongTensor(seq).unsqueeze(0).to(DEVICE)
        beh_tensor = torch.FloatTensor(beh_seq).unsqueeze(0).to(DEVICE)
        
        enc_out = model(seq_tensor, beh_tensor)
        seq_emb = enc_out[:, -1, :]
        
        cand_tensor = torch.LongTensor(candidates).to(DEVICE)
        cand_emb = model.encoder.item_embedding(cand_tensor)
        
        scores = torch.matmul(seq_emb, cand_emb.T).squeeze().cpu().numpy()
        ranked_indices = scores.argsort()[::-1][:20]
        top_items = [candidates[i] for i in ranked_indices if candidates[i] not in rated]
        
        if len(top_items) < 20:
            remaining = 20 - len(top_items)
            top_items.extend([i for i in list(items) if i not in top_items][:remaining])
        
        recommendations[session] = top_items[:20]

print(f"Generated recommendations for {len(recommendations)} sessions")

Generating recommendations: 100%|██████████| 50000/50000 [01:08<00:00, 728.40it/s]

Generated recommendations for 50000 sessions


In [16]:
# Create Submission
print("\n--- Creating Submission ---")

type_labels = ['clicks', 'carts', 'orders']
submission_rows = []

for session in tqdm(recommendations.keys(), desc="Creating submission"):
    recs = recommendations[session]
    labels_str = ' '.join(map(str, recs[:20]))
    for action_type in type_labels:
        submission_rows.append({
            'session_type': f'{session}_{action_type}',
            'labels': labels_str
        })

submission_df = pl.DataFrame(submission_rows)
print(f"Submission shape: {submission_df.shape}")
print(submission_df.head(10))


--- Creating Submission ---


Creating submission: 100%|██████████| 50000/50000 [00:00<00:00, 340666.31it/s]


Submission shape: (150000, 2)
shape: (10, 2)
┌─────────────────┬─────────────────────────────────┐
│ session_type    ┆ labels                          │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ 11098528_clicks ┆ 161690 183670 53503 129988 152… │
│ 11098528_carts  ┆ 161690 183670 53503 129988 152… │
│ 11098528_orders ┆ 161690 183670 53503 129988 152… │
│ 11098529_clicks ┆ 11268 102695 163936 18267 4509… │
│ 11098529_carts  ┆ 11268 102695 163936 18267 4509… │
│ 11098529_orders ┆ 11268 102695 163936 18267 4509… │
│ 11098530_clicks ┆ 19803 79222 33656 166489 71048… │
│ 11098530_carts  ┆ 19803 79222 33656 166489 71048… │
│ 11098530_orders ┆ 19803 79222 33656 166489 71048… │
│ 11098531_clicks ┆ 2241 108204 50122 17612 156463… │
└─────────────────┴─────────────────────────────────┘


In [17]:
# Save Submission
SUBMISSION_PATH = '/kaggle/working/submission.csv'
submission_df.write_csv(SUBMISSION_PATH)

print(f"\nSubmission saved to: {SUBMISSION_PATH}")
print(f"Total rows: {len(submission_df)}")

verify = pl.read_csv(SUBMISSION_PATH)
print("\nSubmission preview:")
print(verify.head(10))


Submission saved to: /kaggle/working/submission.csv
Total rows: 150000

Submission preview:
shape: (10, 2)
┌─────────────────┬─────────────────────────────────┐
│ session_type    ┆ labels                          │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ 11098528_clicks ┆ 161690 183670 53503 129988 152… │
│ 11098528_carts  ┆ 161690 183670 53503 129988 152… │
│ 11098528_orders ┆ 161690 183670 53503 129988 152… │
│ 11098529_clicks ┆ 11268 102695 163936 18267 4509… │
│ 11098529_carts  ┆ 11268 102695 163936 18267 4509… │
│ 11098529_orders ┆ 11268 102695 163936 18267 4509… │
│ 11098530_clicks ┆ 19803 79222 33656 166489 71048… │
│ 11098530_carts  ┆ 19803 79222 33656 166489 71048… │
│ 11098530_orders ┆ 19803 79222 33656 166489 71048… │
│ 11098531_clicks ┆ 2241 108204 50122 17612 156463… │
└─────────────────┴─────────────────────────────────┘


In [18]:
# Final Summary
print("\n" + "="*70)
print("FINAL SUMMARY - MBSRec on Otto Validation Dataset")
print("="*70)
print("\nModel: MBSRec (Multi-Behavioral Sequential Recommendation)")
print("Paper: https://arxiv.org/pdf/2312.09684v1")
print("\nBest Validation Metrics:")
print(f"  Recall@10:  {best_metrics['Recall@10']:.4f}")
print(f"  Recall@20:  {best_metrics['Recall@20']:.4f}")
print(f"  NDCG@10:    {best_metrics['NDCG@10']:.4f}")
print(f"  NDCG@20:    {best_metrics['NDCG@20']:.4f}")
print(f"  HR@10:      {best_metrics['HR@10']:.4f}")
print(f"  HR@20:      {best_metrics['HR@20']:.4f}")
print(f"\nSubmission: {SUBMISSION_PATH}")
print(f"Log file: {LOG_FILE}")
print("="*70)


FINAL SUMMARY - MBSRec on Otto Validation Dataset

Model: MBSRec (Multi-Behavioral Sequential Recommendation)
Paper: https://arxiv.org/pdf/2312.09684v1

Best Validation Metrics:
  Recall@10:  0.0063
  Recall@20:  0.9549
  NDCG@10:    0.0039
  NDCG@20:    0.2601
  HR@10:      0.0063
  HR@20:      0.9549

Submission: /kaggle/working/submission.csv
Log file: /kaggle/working/mbsrec_output/training_log.txt
